# 🧪 Hacklanta Agent Testing

Interactive notebook to test the **AudioMatchAgent** and **StyleDirectorAgent** on sample data.

## 0. Setup — Load API Keys

In [ ]:
import os, sys

# Load keys from secrets.env
with open('secrets.env') as f:
    for line in f:
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            key, val = line.split('=', 1)
            val = val.strip().strip('"').strip("'")  # strip surrounding quotes
            os.environ[key.strip()] = val

API_KEY = os.environ.get('OPENROUTER_API_KEY', '')
ok = API_KEY and API_KEY != 'your-openrouter-api-key-here'
print(f'API Key loaded: {"✅" if ok else "❌ — edit secrets.env first!"}')
if ok:
    print(f'  Key prefix: {API_KEY[:12]}...')

---
## 1. ML Setup Verification
Quick check that all local ML dependencies are importable.

In [ ]:
checks = {}
for name, imp in [
    ('OpenCV', 'cv2'),
    ('NumPy', 'numpy'),
    ('Librosa', 'librosa'),
    ('Ultralytics (YOLO)', 'ultralytics'),
    ('MediaPipe', 'mediapipe'),
    ('Torch', 'torch'),
]:
    try:
        mod = __import__(imp)
        ver = getattr(mod, '__version__', '?')
        checks[name] = f'✅ {ver}'
    except ImportError:
        checks[name] = '❌ not installed'

for k, v in checks.items():
    print(f'{k:25s} {v}')

---
## 2. Test OpenRouter Client (API connectivity)

In [ ]:
from openrouter_client import OpenRouterClient

client = OpenRouterClient(api_key=API_KEY)

response = client.chat.completions.create(
    model='openai/gpt-4o-mini',
    messages=[{'role': 'user', 'content': 'Say hello in exactly 5 words.'}],
    max_tokens=20,
)

print('Model:', response.model)
print('Response:', response.get_content())
print('Tokens used:', response.usage)
client.close()

---
## 3. Test AudioMatchAgent

### 3a. Generate a synthetic test audio file

In [ ]:
import numpy as np
import soundfile as sf

# Generate a 5-second 120 BPM synthetic drum pattern
sr = 22050
duration = 5.0
t = np.linspace(0, duration, int(sr * duration), endpoint=False)

# Simulate beats at 120 BPM (2 beats/sec)
bpm = 120
beat_interval = 60.0 / bpm
signal = np.zeros_like(t)

for beat_time in np.arange(0, duration, beat_interval):
    # Short burst of noise at each beat
    mask = (t >= beat_time) & (t < beat_time + 0.05)
    signal[mask] = np.random.randn(mask.sum()) * 0.8

# Add a bass tone
signal += 0.3 * np.sin(2 * np.pi * 80 * t)

test_audio = '/tmp/test_120bpm.wav'
sf.write(test_audio, signal, sr)
print(f'Created test audio: {test_audio} ({duration}s, target BPM={bpm})')

### 3b. Analyze audio features (LOCAL_ONLY — no API key needed)

In [ ]:
from agents import AudioMatchAgent, ProcessingMode

audio_agent = AudioMatchAgent(
    api_key=API_KEY or 'dummy',
    mode=ProcessingMode.LOCAL_ONLY,
)

features = audio_agent.analyze_audio_file(test_audio)

print('=== Audio Features ===')
for k, v in features.to_dict().items():
    print(f'  {k:25s}: {v}')

### 3c. Classify genre/mood (LOCAL_ONLY)

In [ ]:
genre_result = audio_agent.classify_genre_mood(test_audio)

print('=== Genre/Mood Classification (Local) ===')
print(f'  Genres:     {genre_result.genres}')
print(f'  Moods:      {genre_result.moods}')
print(f'  Confidence: {genre_result.confidence:.2f}')
print(f'  Source:     {genre_result.source}')

### 3d. Classify genre/mood (HYBRID — API fallback if confidence is low)

In [ ]:
# Only run this cell if you have a valid API key
if API_KEY and API_KEY != 'your-openrouter-api-key-here':
    hybrid_agent = AudioMatchAgent(
        api_key=API_KEY,
        mode=ProcessingMode.HYBRID,
    )
    hybrid_result = hybrid_agent.classify_genre_mood(test_audio)
    print('=== Genre/Mood Classification (Hybrid) ===')
    print(f'  Genres:     {hybrid_result.genres}')
    print(f'  Moods:      {hybrid_result.moods}')
    print(f'  Confidence: {hybrid_result.confidence:.2f}')
    print(f'  Source:     {hybrid_result.source}')
    print(f'  Fallback:   {hybrid_result.fallback_triggered}')
else:
    print('⚠️  Skipping — no valid API key in secrets.env')

---
## 4. Test StyleDirectorAgent

### 4a. Generate synthetic test images

In [ ]:
import cv2

# Warm sunset image: gradient from orange to deep red
h, w = 200, 300
warm_img = np.zeros((h, w, 3), dtype=np.uint8)
for y in range(h):
    ratio = y / h
    warm_img[y, :] = [int(30 * ratio), int(80 + 100 * (1 - ratio)), int(200 + 55 * (1 - ratio))]  # BGR

# Cool blue image: gradient from cyan to dark blue
cool_img = np.zeros((h, w, 3), dtype=np.uint8)
for y in range(h):
    ratio = y / h
    cool_img[y, :] = [int(200 + 55 * (1 - ratio)), int(150 * (1 - ratio)), int(30 * ratio)]  # BGR

warm_path = '/tmp/test_warm_image.jpg'
cool_path = '/tmp/test_cool_image.jpg'
cv2.imwrite(warm_path, warm_img)
cv2.imwrite(cool_path, cool_img)

# Display
from IPython.display import display, Image as IPImage
print('Warm image:'); display(IPImage(filename=warm_path, width=300))
print('Cool image:'); display(IPImage(filename=cool_path, width=300))

### 4b. Analyze style (LOCAL_ONLY)

In [ ]:
from agents import StyleDirectorAgent, ProcessingMode

style_agent = StyleDirectorAgent(
    api_key=API_KEY or 'dummy',
    mode=ProcessingMode.LOCAL_ONLY,
)

print('=== Warm Image Analysis ===')
warm_result = style_agent.analyze_style(warm_path)
for k, v in warm_result.to_dict().items():
    if k not in ('target_image_histogram', 'source_image_histogram'):
        print(f'  {k:25s}: {v}')

print('\n=== Cool Image Analysis ===')
cool_result = style_agent.analyze_style(cool_path)
for k, v in cool_result.to_dict().items():
    if k not in ('target_image_histogram', 'source_image_histogram'):
        print(f'  {k:25s}: {v}')

### 4c. Compare two images (histogram similarity)

In [ ]:
comparison = style_agent.analyze_style(warm_path, reference_path=cool_path)

print(f'Histogram similarity (warm vs cool): {comparison.histogram_similarity:.3f}')
print(f'(0 = completely different, 1 = identical)')

### 4d. Style transfer (histogram matching)

In [ ]:
output_path = style_agent.apply_style_transfer(warm_path, cool_path)

if output_path:
    print(f'Style transfer output: {output_path}')
    display(IPImage(filename=output_path, width=300))
else:
    print('Style transfer failed')

### 4e. LUT Recommendations

In [ ]:
luts = style_agent.get_lut_recommendations(warm_result)

print(f'=== LUT Recommendations for warm image (mood: {warm_result.mood}) ===')
for i, lut in enumerate(luts, 1):
    print(f'\n  {i}. {lut.name} ({lut.category})')
    print(f'     {lut.description}')
    print(f'     Confidence: {lut.confidence:.2f} | Intensity: {lut.intensity:.2f}')
    print(f'     Notes: {lut.color_grading_notes}')

---
## 5. Test with your own files

Replace the paths below with your own audio/image files.

In [ ]:
# === YOUR AUDIO FILE ===
# my_audio = 'path/to/your/song.mp3'
# features = audio_agent.analyze_audio_file(my_audio)
# print(features.to_dict())
# result = audio_agent.classify_genre_mood(my_audio)
# print(result.genres, result.moods)

# === YOUR IMAGE FILE ===
# my_image = 'path/to/your/photo.jpg'
# result = style_agent.analyze_style(my_image)
# print(result.to_dict())